In [ ]:
!pip install snowflake-connector-python xgboost scikit-learn pandas numpy joblib -q
print("✅ Libraries installed!")

✅ Libraries installed!


In [ ]:
import snowflake.connector
import pandas as pd

# ⚠️ Replace with your credentials
conn = snowflake.connector.connect(
    account   = "OUFSWVG-OZ74799",  # org-account format
    user      = "PRASADPATIL0103",
    password  = "Applepie@01032202",    # ⚠️ your actual password
    database  = "NYC_TLC",
    schema    = "MARTS",
    warehouse = "COMPUTE_WH",
    host      = "OUFSWVG-OZ74799.snowflakecomputing.com"  # ← add this line
)

print("✅ Connected to Snowflake!")

# Load FACT_TRIPS — sample 500K rows for faster training
query = """
    SELECT
        TRIP_DISTANCE,
        PICKUP_LOCATION_ID,
        DROPOFF_LOCATION_ID,
        PICKUP_HOUR,
        PICKUP_DAY,
        PICKUP_MONTH,
        PASSENGER_COUNT,
        FARE_AMOUNT,
        TIP_AMOUNT,
        TOTAL_AMOUNT,
        TRIP_DURATION_MIN,
        REVENUE_PER_MILE,
        TIP_PERCENTAGE,
        IS_WEEKEND,
        PAYMENT_TYPE,
        TRIP_CATEGORY,
        TIME_OF_DAY
    FROM NYC_TLC.MARTS.FACT_TRIPS
    SAMPLE (500000 ROWS)
"""

print("⏳ Loading data from Snowflake...")
df = pd.read_sql(query, conn)
conn.close()

print(f"✅ Data loaded!")
print(f"📊 Rows    : {len(df):,}")
print(f"📋 Columns : {len(df.columns)}")
df.head(3)

✅ Connected to Snowflake!
⏳ Loading data from Snowflake...


/tmp/ipython-input-1752/1258681345.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


✅ Data loaded!
📊 Rows    : 500,000
📋 Columns : 17


,TRIP_DISTANCE,PICKUP_LOCATION_ID,DROPOFF_LOCATION_ID,PICKUP_HOUR,PICKUP_DAY,PICKUP_MONTH,PASSENGER_COUNT,FARE_AMOUNT,TIP_AMOUNT,TOTAL_AMOUNT,TRIP_DURATION_MIN,REVENUE_PER_MILE,TIP_PERCENTAGE,IS_WEEKEND,PAYMENT_TYPE,TRIP_CATEGORY,TIME_OF_DAY
0,1.41,166,75,11,7,1,1,11.4,0.00,12.90,10.03,9.15,0.00,True,1,Medium,Off Peak
1,3.40,163,41,23,4,1,1,17.0,0.00,22.00,15.12,6.47,0.00,False,2,Medium,Late Night
2,1.02,161,107,13,4,1,1,9.3,2.66,15.96,8.15,15.65,16.67,False,1,Medium,Off Peak


In [ ]:
# Reload data fresh and check nulls BEFORE any encoding
print("Shape:", df.shape)
print("\nNull counts:")
print(df.isnull().sum())
print("\nSample TRIP_CATEGORY values:")
print(df["TRIP_CATEGORY"].value_counts())
print("\nSample TIME_OF_DAY values:")
print(df["TIME_OF_DAY"].value_counts())

Shape: (500000, 17)

Null counts:
TRIP_DISTANCE          0
PICKUP_LOCATION_ID     0
DROPOFF_LOCATION_ID    0
PICKUP_HOUR            0
PICKUP_DAY             0
PICKUP_MONTH           0
PASSENGER_COUNT        0
FARE_AMOUNT            0
TIP_AMOUNT             0
TOTAL_AMOUNT           0
TRIP_DURATION_MIN      0
REVENUE_PER_MILE       0
TIP_PERCENTAGE         0
IS_WEEKEND             0
PAYMENT_TYPE           0
TRIP_CATEGORY          0
TIME_OF_DAY            0
dtype: int64

Sample TRIP_CATEGORY values:
TRIP_CATEGORY
Medium       306368
Short        114775
Long          56457
Very Long     22400
Name: count, dtype: int64

Sample TIME_OF_DAY values:
TIME_OF_DAY
Off Peak        226828
Evening Rush    134874
Late Night       82370
Morning Rush     55928
Name: count, dtype: int64


In [ ]:
import numpy as np

# Fix types — fill NaN before casting to int
df["IS_WEEKEND"]    = df["IS_WEEKEND"].astype(int)
df["HIGH_TIP"]      = (df["TIP_AMOUNT"] > 3).astype(int)

df["TRIP_CATEGORY"] = pd.to_numeric(df["TRIP_CATEGORY"], errors="coerce").fillna(1).astype(int)
df["TIME_OF_DAY"]   = pd.to_numeric(df["TIME_OF_DAY"], errors="coerce").fillna(1).astype(int)

# Drop any remaining nulls in key columns
df = df.dropna(subset=[
    "TRIP_DISTANCE", "FARE_AMOUNT", "TOTAL_AMOUNT",
    "PICKUP_HOUR", "TRIP_DURATION_MIN", "REVENUE_PER_MILE"
])

print(f"✅ Feature engineering complete!")
print(f"📊 Clean rows : {len(df):,}")
print(f"🏷️  High tip % : {df['HIGH_TIP'].mean()*100:.1f}%")
print(f"\nDtypes:")
print(df[["TRIP_CATEGORY", "TIME_OF_DAY", "IS_WEEKEND", "HIGH_TIP"]].dtypes)


✅ Feature engineering complete!
📊 Clean rows : 500,000
🏷️  High tip % : 43.5%

Dtypes:
TRIP_CATEGORY    int64
TIME_OF_DAY      int64
IS_WEEKEND       int64
HIGH_TIP         int64
dtype: object


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd # Ensure pandas is imported for DataFrame checks

# Features and target
FARE_FEATURES = [
    "TRIP_DISTANCE", "PICKUP_LOCATION_ID", "DROPOFF_LOCATION_ID",
    "PICKUP_HOUR", "PICKUP_DAY", "PICKUP_MONTH",
    "PASSENGER_COUNT", "IS_WEEKEND", "TRIP_CATEGORY", "TIME_OF_DAY"
]

X = df[FARE_FEATURES]
y = df["TOTAL_AMOUNT"]

# --- Start of proposed change ---
if X.empty:
    print("❌ Error: The DataFrame 'df' is empty after preprocessing in the previous cell.")
    print("No data available to train the Fare Prediction model.")
    print("Please check the data loading and preprocessing steps (especially `df.dropna()`) in the preceding cells.")
else:
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Train model
    print("⏳ Training Fare Prediction model...")
    fare_model = XGBRegressor(
        n_estimators    = 200,
        max_depth       = 6,
        learning_rate   = 0.1,
        subsample       = 0.8,
        random_state    = 42,
        n_jobs          = -1
    )
    fare_model.fit(X_train, y_train)

    # Evaluate
    y_pred = fare_model.predict(X_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    r2     = r2_score(y_test, y_pred)

    print(f"\n✅ Fare Prediction Model trained!")
    print(f"   RMSE : ${rmse:.2f}")
    print(f"   R²   : {r2:.4f}")

    # Feature importance
    feat_imp = pd.DataFrame({
        "feature"   : FARE_FEATURES,
        "importance": fare_model.feature_importances_
    }).sort_values("importance", ascending=False)
    print(f"\n📊 Top 5 Features:\n{feat_imp.head()}")
# --- End of proposed change ---


⏳ Training Fare Prediction model...

✅ Fare Prediction Model trained!
   RMSE : $6.61
   R²   : 0.9047

📊 Top 5 Features:
               feature  importance
0        TRIP_DISTANCE    0.862639
2  DROPOFF_LOCATION_ID    0.052132
7           IS_WEEKEND    0.015625
1   PICKUP_LOCATION_ID    0.015027
3          PICKUP_HOUR    0.014910


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Aggregate trips by hour/day/location
demand_df = df.groupby([
    "PICKUP_LOCATION_ID", "PICKUP_HOUR",
    "PICKUP_DAY", "PICKUP_MONTH", "IS_WEEKEND"
]).size().reset_index(name="TRIP_COUNT")

print(f"📊 Demand dataset: {len(demand_df):,} rows")

# Features and target
DEMAND_FEATURES = [
    "PICKUP_LOCATION_ID", "PICKUP_HOUR",
    "PICKUP_DAY", "PICKUP_MONTH", "IS_WEEKEND"
]

X = demand_df[DEMAND_FEATURES]
y = demand_df["TRIP_COUNT"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
print("⏳ Training Demand Forecasting model...")
demand_model = XGBRegressor(
    n_estimators  = 200,
    max_depth     = 5,
    learning_rate = 0.1,
    random_state  = 42,
    n_jobs        = -1
)
demand_model.fit(X_train, y_train)

# Evaluate
y_pred = demand_model.predict(X_test)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
r2     = r2_score(y_test, y_pred)

print(f"\n✅ Demand Forecasting Model trained!")
print(f"   RMSE : {rmse:.2f} trips")
print(f"   R²   : {r2:.4f}")

📊 Demand dataset: 25,548 rows
⏳ Training Demand Forecasting model...

✅ Demand Forecasting Model trained!
   RMSE : 9.22 trips
   R²   : 0.8956


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Filter credit card only
tip_df = df[df["PAYMENT_TYPE"] == 1].copy()

# New target: tip amount > $3 = generous tip
# This captures actual behavior better than percentage
tip_df["HIGH_TIP"] = (tip_df["TIP_AMOUNT"] > 3).astype(int)

print(f"📊 Tip dataset: {len(tip_df):,} rows")
print(f"🏷️  High tip rate: {tip_df['HIGH_TIP'].mean()*100:.1f}%")

# Check distribution
print(f"\nTip amount distribution:")
print(tip_df["TIP_AMOUNT"].describe())

TIP_FEATURES = [
    "TRIP_DISTANCE", "FARE_AMOUNT", "PICKUP_HOUR",
    "PICKUP_DAY", "IS_WEEKEND", "TRIP_CATEGORY",
    "TIME_OF_DAY", "PICKUP_LOCATION_ID",
    "TRIP_DURATION_MIN", "REVENUE_PER_MILE"
]

X = tip_df[TIP_FEATURES]
y = tip_df["HIGH_TIP"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\n⏳ Training Tip Prediction model...")
tip_model = XGBClassifier(
    n_estimators     = 300,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    random_state     = 42,
    n_jobs           = -1
)
tip_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = tip_model.predict(X_test)
acc    = accuracy_score(y_test, y_pred)

print(f"\n✅ Tip Prediction Model trained!")
print(f"   Accuracy : {acc*100:.1f}%")
print(f"\n📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Low Tip", "High Tip"]))

📊 Tip dataset: 399,994 rows
🏷️  High tip rate: 53.1%

Tip amount distribution:
count    399994.000000
mean          4.155587
std           3.821125
min           0.000000
25%           2.150000
50%           3.150000
75%           4.680000
max         300.000000
Name: TIP_AMOUNT, dtype: float64

⏳ Training Tip Prediction model...

✅ Tip Prediction Model trained!
   Accuracy : 90.5%

📊 Classification Report:
              precision    recall  f1-score   support

     Low Tip       0.92      0.87      0.90     37520
    High Tip       0.89      0.94      0.91     42479

    accuracy                           0.91     79999
   macro avg       0.91      0.90      0.90     79999
weighted avg       0.91      0.91      0.90     79999



In [ ]:
import joblib
import os

os.makedirs("/content/models", exist_ok=True)

joblib.dump(fare_model,   "/content/models/fare_model.pkl")
joblib.dump(demand_model, "/content/models/demand_model.pkl")
joblib.dump(tip_model,    "/content/models/tip_model.pkl")

# Save feature lists too — needed by FastAPI
import json
feature_config = {
    "fare_features"   : FARE_FEATURES,
    "demand_features" : DEMAND_FEATURES,
    "tip_features"    : TIP_FEATURES
}
with open("/content/models/feature_config.json", "w") as f:
    json.dump(feature_config, f)

print("✅ All models saved!")
for f in os.listdir("/content/models"):
    size_kb = os.path.getsize(f"/content/models/{f}") / 1024
    print(f"  📄 {f} — {size_kb:.1f} KB")

✅ All models saved!
  📄 tip_model.pkl — 1428.2 KB
  📄 demand_model.pkl — 542.3 KB
  📄 feature_config.json — 0.5 KB
  📄 fare_model.pkl — 891.5 KB


In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import storage

PROJECT_ID    = "nyc-tlc-pipeline"
BRONZE_BUCKET = "nyc-tlc-bronze-prasad"  # ⚠️ reuse your bronze bucket

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BRONZE_BUCKET)

for filename in os.listdir("/content/models"):
    local_path = f"/content/models/{filename}"
    gcs_path   = f"models/{filename}"
    blob       = bucket.blob(gcs_path)
    blob.upload_from_filename(local_path)
    print(f"☁️  Uploaded: {gcs_path}")

print("\n✅ All models uploaded to GCS!")

☁️  Uploaded: models/tip_model.pkl
☁️  Uploaded: models/demand_model.pkl
☁️  Uploaded: models/feature_config.json
☁️  Uploaded: models/fare_model.pkl

✅ All models uploaded to GCS!


In [ ]:
import numpy as np

# Test fare prediction
sample_trip = pd.DataFrame([{
    "TRIP_DISTANCE"        : 3.5,
    "PICKUP_LOCATION_ID"   : 161,
    "DROPOFF_LOCATION_ID"  : 237,
    "PICKUP_HOUR"          : 8,
    "PICKUP_DAY"           : 2,
    "PICKUP_MONTH"         : 1,
    "PASSENGER_COUNT"      : 1,
    "IS_WEEKEND"           : 0,
    "TRIP_CATEGORY"        : 1,
    "TIME_OF_DAY"          : 2
}])

fare_pred   = fare_model.predict(sample_trip)[0]
print(f"🚕 Sample Trip Prediction:")
print(f"   Distance     : 3.5 miles")
print(f"   Pickup hour  : 8am (Morning Rush)")
print(f"   Predicted fare: ${fare_pred:.2f}")

# Test tip prediction — add missing features
sample_tip = pd.DataFrame([{
    "TRIP_DISTANCE"       : 3.5,
    "FARE_AMOUNT"         : fare_pred,
    "PICKUP_HOUR"         : 8,
    "PICKUP_DAY"          : 2,
    "IS_WEEKEND"          : 0,
    "TRIP_CATEGORY"       : 1,
    "TIME_OF_DAY"         : 2,
    "PICKUP_LOCATION_ID"  : 161,
    "TRIP_DURATION_MIN"   : 12.5,   # ← added
    "REVENUE_PER_MILE"    : fare_pred / 3.5  # ← added
}])

tip_prob  = tip_model.predict_proba(sample_tip)[0][1]
tip_label = "High Tip 💰" if tip_prob > 0.5 else "Low Tip"
print(f"\n💳 Tip Prediction:")
print(f"   Probability of high tip: {tip_prob*100:.1f}%")
print(f"   Prediction: {tip_label}")

🚕 Sample Trip Prediction:
   Distance     : 3.5 miles
   Pickup hour  : 8am (Morning Rush)
   Predicted fare: $29.91

💳 Tip Prediction:
   Probability of high tip: 79.5%
   Prediction: High Tip 💰
